# 4. Wavelet Transform pipeline

## 1. Import libraries

In [5]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Path settings
import sys
sys.path.append('..')

# import basic lib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# import scikit learn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold, GroupKFold

# import mne
import mne

from src.preprocessing import preprocess_eeg_data, preprocess_subject_runs
from src.evaluation import evaluate_cross_validation_baseline
from src.bonus_pipeline import create_wavelet_pipeline, create_wavelet_select_pipeline


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 2. Wrapper function for GroupKFold

In [6]:
def preprocess_subject_runs_with_groups(
        subject_id: int,
        run_ids: list[int]
    ):
    """
    Preprocess the EEG data for a given subject and runs, just for this task
    """
    X_list, y_list, groups = [], [], []

    for run_id in run_ids:
        X_runs, y_runs = preprocess_subject_runs(subject_id, [run_id])
        X_list.append(X_runs)
        y_list.append(y_runs)
        groups.extend([run_id] * len(y_runs))

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    groups = np.array(groups)

    return X, y, groups

## 3. Wavelet transform pipeline

In [8]:
np.random.seed(42)
sns.set_theme(style='whitegrid', context='notebook')

X, y, groups = preprocess_subject_runs_with_groups(subject_id=4, run_ids=[6, 10, 14])

cv = GroupKFold(n_splits=3)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"groups shape: {groups.shape}")

pipeline = create_wavelet_pipeline()
select_pipeline = create_wavelet_select_pipeline(k=20)
baseline_scores = cross_val_score(
    pipeline, 
    X,
    y, 
    cv=cv, 
    groups=groups
)

select_scores = cross_val_score(
    select_pipeline,
    X,
    y,
    cv=cv,
    groups=groups,
    error_score='raise'
)

print(baseline_scores)
print(baseline_scores.mean())
print(select_scores)
print(select_scores.mean())

print("X shape:", X.shape)
print("y shape:", y.shape)
print("label count:", np.bincount(y))

features = pipeline.named_steps["wavelet"].fit_transform(X, y)

print("Wavelet baseline:", baseline_scores, baseline_scores.mean())
print("Wavelet + SelectKBest:", select_scores, select_scores.mean())

Extracting EDF parameters from /home/donghank/total-perspective-vortex/notebook/../physionet.org/files/eegmmidb/1.0.0/S004/S004R06.edf...
Setting channel info structure...
Creating raw.info structure...


Reading 0 ... 19679  =      0.000 ...   122.994 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 265 samples (1.656 s)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Not setting metadata
15 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 15 events and 641 original time points ...
0 bad epochs dropped
Extracting EDF parameters from /home/donghank/total-perspective-vortex/notebook/../physionet.org/files/